# 1.1 Persiapan dan Perkenalan Lingkungan

In [2]:
from neo4j import GraphDatabase

print("Neo4j Driver berhasil diimpor!")

Neo4j Driver berhasil diimpor!


In [7]:
from neo4j import GraphDatabase

NEO4J_URI = "neo4j://127.0.0.1:7687"
NEO4J_USERNAME = "neo4j"
NEO4J_PASSWORD = input("Masukkan password Neo4j lokal: ")

driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USERNAME, NEO4J_PASSWORD)
)

driver.verify_connectivity()
print("Koneksi ke Neo4j lokal berhasil!")

Koneksi ke Neo4j lokal berhasil!


Environment Python dibuat menggunakan `venv` dengan nama `myvenv` dan Python versi 3.10.19. `pip` telah disiapkan untuk mengelola paket, dan Neo4j Python Driver versi 6.4.1 berhasil diimpor.

Dataset RetailRocket tersedia di folder `Dataset`, terdiri dari `events.csv`, `category_tree.csv`, `item_properties_part1.csv`, dan `item_properties_part2.csv`.

Koneksi Python ke database Neo4j lokal berhasil diverifikasi, sehingga lingkungan siap digunakan untuk tahap selanjutnya.

# 1.2 — Mempelajari Penggunaan Driver Neo4j

1. Membuat node dan relationship

In [9]:
query = """
MERGE (a:Person {name: 'Arthur'})
MERGE (g:Person {name: 'Guinevere'})
MERGE (l:Person {name: 'Lancelot'})
MERGE (m:Person {name: 'Merlin'})

MERGE (a)-[:KNOWS]->(g)
MERGE (a)-[:KNOWS]->(l)
MERGE (g)-[:KNOWS]->(m)
MERGE (l)-[:KNOWS]->(m)

RETURN a.name AS orang, g.name AS teman
"""

with driver.session() as session:
    result = session.run(query)
    for record in result:
        print(record)

<Record orang='Arthur' teman='Guinevere'>


In [10]:
query = """
MATCH (p:Person)
RETURN p.name AS nama
ORDER BY nama
"""

with driver.session() as session:
    result = session.run(query)
    for record in result:
        print(record["nama"])

Arthur
Guinevere
Lancelot
Merlin


In [11]:
query = """
MATCH (a:Person)-[r:KNOWS]->(b:Person)
RETURN a.name AS orang, b.name AS teman
ORDER BY orang, teman
"""

with driver.session() as session:
    result = session.run(query)
    for record in result:
        print(f"{record['orang']} -> {record['teman']}")

Arthur -> Guinevere
Arthur -> Lancelot
Guinevere -> Merlin
Lancelot -> Merlin


# 1.3 — Impor Data RetailRocket ke Neo4j.

1. impor data

In [14]:
import pandas as pd

events_path = "Dataset/events.csv"
df_events = pd.read_csv(events_path)

print("Jumlah baris:", len(df_events))
print("Kolom:", df_events.columns.tolist())
print(df_events.head())

Jumlah baris: 2756101
Kolom: ['timestamp', 'visitorid', 'event', 'itemid', 'transactionid']
       timestamp  visitorid event  itemid  transactionid
0  1433221332117     257597  view  355908            NaN
1  1433224214164     992329  view  248676            NaN
2  1433221999827     111016  view  318965            NaN
3  1433221955914     483717  view  253185            NaN
4  1433221337106     951259  view  367447            NaN


In [15]:
print("Jumlah nilai kosong per kolom:")
print(df_events.isna().sum())

print("\nJumlah setiap jenis event:")
print(df_events["event"].value_counts())

print("\nJumlah visitor unik:", df_events["visitorid"].nunique())
print("Jumlah item unik:", df_events["itemid"].nunique())

Jumlah nilai kosong per kolom:
timestamp              0
visitorid              0
event                  0
itemid                 0
transactionid    2733644
dtype: int64

Jumlah setiap jenis event:
event
view           2664312
addtocart        69332
transaction      22457
Name: count, dtype: int64

Jumlah visitor unik: 1407580
Jumlah item unik: 235061


In [16]:
with driver.session() as session:
    session.run("""
        CREATE CONSTRAINT visitor_id_unique IF NOT EXISTS
        FOR (v:Visitor)
        REQUIRE v.visitorId IS UNIQUE
    """)

    session.run("""
        CREATE CONSTRAINT item_id_unique IF NOT EXISTS
        FOR (i:Item)
        REQUIRE i.itemId IS UNIQUE
    """)

print("Constraint Visitor dan Item berhasil diperiksa/dibuat.")

Constraint Visitor dan Item berhasil diperiksa/dibuat.


2. Impor event ke Neo4j secara bertahap

In [17]:
BATCH_SIZE = 10_000

query = """
UNWIND $batch AS row

MERGE (v:Visitor {visitorId: row.visitorid})
MERGE (i:Item {itemId: row.itemid})

WITH v, i, row
CALL (v, i, row) {
    WITH v, i, row
    WHERE row.event = 'view'
    CREATE (v)-[:VIEWED {timestamp: row.timestamp}]->(i)
    RETURN 1 AS created

    UNION

    WITH v, i, row
    WHERE row.event = 'addtocart'
    CREATE (v)-[:ADDED_TO_CART {timestamp: row.timestamp}]->(i)
    RETURN 1 AS created

    UNION

    WITH v, i, row
    WHERE row.event = 'transaction'
    CREATE (v)-[:BOUGHT {
        timestamp: row.timestamp,
        transactionId: row.transactionid
    }]->(i)
    RETURN 1 AS created
}
RETURN count(*) AS processed
"""

total = len(df_events)

for start in range(0, total, BATCH_SIZE):
    batch_df = df_events.iloc[start:start + BATCH_SIZE]

    batch = (
        batch_df.astype(object)
        .where(pd.notna(batch_df), None)
        .to_dict(orient="records")
    )

    with driver.session() as session:
        result = session.run(query, batch=batch)
        processed = result.single()["processed"]

    print(f"Progress: {min(start + len(batch), total):,}/{total:,}")

print("Impor RetailRocket selesai.")

Progress: 10,000/2,756,101
Progress: 20,000/2,756,101
Progress: 30,000/2,756,101
Progress: 40,000/2,756,101
Progress: 50,000/2,756,101
Progress: 60,000/2,756,101
Progress: 70,000/2,756,101
Progress: 80,000/2,756,101
Progress: 90,000/2,756,101
Progress: 100,000/2,756,101
Progress: 110,000/2,756,101
Progress: 120,000/2,756,101
Progress: 130,000/2,756,101
Progress: 140,000/2,756,101
Progress: 150,000/2,756,101
Progress: 160,000/2,756,101
Progress: 170,000/2,756,101
Progress: 180,000/2,756,101
Progress: 190,000/2,756,101
Progress: 200,000/2,756,101
Progress: 210,000/2,756,101
Progress: 220,000/2,756,101
Progress: 230,000/2,756,101
Progress: 240,000/2,756,101
Progress: 250,000/2,756,101
Progress: 260,000/2,756,101
Progress: 270,000/2,756,101
Progress: 280,000/2,756,101
Progress: 290,000/2,756,101
Progress: 300,000/2,756,101
Progress: 310,000/2,756,101
Progress: 320,000/2,756,101
Progress: 330,000/2,756,101
Progress: 340,000/2,756,101
Progress: 350,000/2,756,101
Progress: 360,000/2,756,101
P

In [18]:
query = """
MATCH (v:Visitor)
WITH count(v) AS totalVisitor
MATCH (i:Item)
WITH totalVisitor, count(i) AS totalItem
MATCH ()-[r]->()
RETURN totalVisitor, totalItem, count(r) AS totalRelationship
"""

with driver.session() as session:
    result = session.run(query)
    record = result.single()

    print("Total Visitor:", record["totalVisitor"])
    print("Total Item:", record["totalItem"])
    print("Total Relationship:", record["totalRelationship"])

Total Visitor: 1407580
Total Item: 235061
Total Relationship: 2756105


### Hasil Impor Data RetailRocket

Dataset RetailRocket berhasil diimpor ke Neo4j menggunakan proses batch sebanyak 10.000 baris per batch. Hasil verifikasi menunjukkan terdapat 1.407.580 node `Visitor`, 235.061 node `Item`, dan 2.756.101 relationship aktivitas. Total relationship dalam database adalah 2.756.105, termasuk empat relationship `KNOWS` dari latihan sebelumnya. Dengan demikian, jumlah data hasil impor sesuai dengan jumlah event pada dataset sumber.